# Módulo de visualização e insights para identificação de desertos médicos Sus

**Autora:** Vanessa Batista ([@vandromedae](https://github.com/vandromedae))  
**Repositório:** [github.com/vandromedae/desertos-medicos-sus](https://github.com/vandromedae/desertos-medicos-sus)  
**Licença:** [MIT](https://github.com/vandromedae/desertos-medicos-sus/blob/main/LICENSE)

---

In [ ]:
# ============================================================
#  Desertos Médicos SUS - Visualização e Insights
# ============================================================


import sys
from pathlib import Path

# Adicionar raiz do projeto ao path
notebook_path = Path().resolve()
project_root = notebook_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins

from src.config import (
    DATA_PROCESSED, DATA_EXTERNAL, OUTPUT_MAPAS, OUTPUT_RELATORIOS,
    IBGE_SP_CAPITAL, LIMIAR_DENSIDADE_MEDICA,
)

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Garantir diretórios de saída
OUTPUT_MAPAS.mkdir(parents=True, exist_ok=True)
OUTPUT_RELATORIOS.mkdir(parents=True, exist_ok=True)

print(" Ambiente configurado!")
print(f" Raiz do projeto: {project_root}")
print(f" Mapas de saída: {OUTPUT_MAPAS}")

In [ ]:
# ============================================================
# Carregar todos os dados processados
# ============================================================

print("=" * 70)
print("CARREGANDO DADOS PROCESSADOS")
print("=" * 70)

# 1. Base municipal
arquivo_municipal = DATA_PROCESSED / "base_municipal_para_mapa.parquet"
if not arquivo_municipal.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo_municipal}")
df_municipal = pd.read_parquet(arquivo_municipal)
print(f" Base municipal: {len(df_municipal):,} municípios")

# 2. Setores com acessibilidade (com polígonos reais)
arquivo_setores = DATA_PROCESSED / "setores_com_acessibilidade_real.parquet"
if not arquivo_setores.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo_setores}")
df_setores = pd.read_parquet(arquivo_setores)
print(f" Setores com acessibilidade: {len(df_setores):,} setores")

# 3. CNES agregados
arquivo_cnes = DATA_PROCESSED / "cnes_agregados.parquet"
if not arquivo_cnes.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo_cnes}")
df_cnes = pd.read_parquet(arquivo_cnes)
print(f" CNES agregados: {len(df_cnes):,} estabelecimentos")

# 4. Comparação municipal vs acessibilidade
arquivo_comparacao = DATA_PROCESSED / "comparacao_municipal_vs_e2sfca.parquet"
if arquivo_comparacao.exists():
    df_comparacao = pd.read_parquet(arquivo_comparacao)
    print(f" Comparação municipal vs acessibilidade: {len(df_comparacao):,} municípios")
else:
    df_comparacao = None
    print(" Arquivo de comparação não encontrado")

# 5. Shapefile de municípios de SP
shapefile_municipios = DATA_EXTERNAL / "SP_Municipios_2025" / "SP_Municipios_2025.shp"
if shapefile_municipios.exists():
    gdf_municipios = gpd.read_file(shapefile_municipios)
    gdf_municipios['cod_mun_6'] = gdf_municipios['CD_MUN'].astype(str).str[:6]
    print(f" Shapefile de municípios: {len(gdf_municipios):,} polígonos")
else:
    gdf_municipios = None
    print(" Shapefile de municípios não encontrado - mapa coroplético indisponível")

# 6. Shapefile de setores censitários
shapefile_setores = DATA_EXTERNAL / "SP_setores_CD2022_IBGE" / "SP_setores_CD2022.shp"
if shapefile_setores.exists():
    gdf_setores_geo = gpd.read_file(shapefile_setores)
    print(f" Shapefile de setores: {len(gdf_setores_geo):,} polígonos")
else:
    gdf_setores_geo = None
    print(" Shapefile de setores não encontrado")

print(f"\n Resumo dos dados:")
print(f"   Municípios: {len(df_municipal):,}")
print(f"   Setores censitários: {len(df_setores):,}")
print(f"   CNES: {len(df_cnes):,}")
print(f"   População total: {df_municipal['populacao'].sum():,}")
print(f"   Médicos SUS: {df_municipal['total_medicos'].sum():,}")

In [ ]:
# ============================================================
# MAPA 1: Densidade Médica por Município (Coroplético)
# ============================================================


print("=" * 70)
print("GERANDO MAPA 1: DENSIDADE MÉDICA MUNICIPAL")
print("=" * 70)

if gdf_municipios is None:
    print(" Shapefile de municípios não disponível - pulando mapa 1")
else:
    # 1. Detectar colunas disponíveis no DataFrame municipal
    print("\n Detectando colunas da base municipal...")
    colunas_mun = df_municipal.columns.tolist()
    
    # Detectar coluna de código de município (6 dígitos)
    col_cod_mun = next((c for c in colunas_mun if c in ['cod_mun_6', 'cod_mun_ibge', 'cd_mun_6', 'cod_mun']), None)
    col_nm_mun = next((c for c in colunas_mun if c in ['nm_mun', 'nome_mun', 'municipio']), None)
    col_pop = next((c for c in colunas_mun if c in ['populacao', 'pop', 'v0001']), None)
    col_medicos = next((c for c in colunas_mun if c in ['total_medicos', 'medicos']), None)
    col_densidade = next((c for c in colunas_mun if c in ['medicos_por_1k', 'densidade']), None)
    
    print(f"    Código município: {col_cod_mun}")
    print(f"    Nome município: {col_nm_mun}")
    print(f"    População: {col_pop}")
    print(f"    Total médicos: {col_medicos}")
    print(f"    Densidade: {col_densidade}")
    
    # 2. Criar coluna cod_mun_6 se não existir
    if col_cod_mun and col_cod_mun != 'cod_mun_6':
        df_municipal['cod_mun_6'] = df_municipal[col_cod_mun].astype(str).str[:6]
        print(f"    Criada 'cod_mun_6' a partir de '{col_cod_mun}'")
    elif not col_cod_mun:
        if 'cd_mun' in colunas_mun:
            df_municipal['cod_mun_6'] = df_municipal['cd_mun'].astype(str).str[:6]
            print(f"    Criada 'cod_mun_6' a partir de 'cd_mun'")
        else:
            raise ValueError("Não foi possível identificar coluna de código de município!")
    
    # 3. Merge shapefile + dados
    print("\n Fazendo merge shapefile + dados municipais...")
    
    cols_para_merge = ['cod_mun_6']
    if col_nm_mun: cols_para_merge.append(col_nm_mun)
    if col_pop: cols_para_merge.append(col_pop)
    if col_medicos: cols_para_merge.append(col_medicos)
    if col_densidade: cols_para_merge.append(col_densidade)
    
    gdf_mapa_mun = gdf_municipios.merge(
        df_municipal[cols_para_merge],
        on='cod_mun_6',
        how='left'
    )
    
    # Tratar nulos
    if col_medicos:
        gdf_mapa_mun[col_medicos] = gdf_mapa_mun[col_medicos].fillna(0).astype(int)
    if col_densidade:
        gdf_mapa_mun[col_densidade] = gdf_mapa_mun[col_densidade].fillna(0)
    
    print(f"    Merge concluído: {len(gdf_mapa_mun):,} municípios")
    
    # 4. Classificar densidade
    def classificar_densidade(valor):
        if valor < 1.0:
            return '1. Crítico (<1,0)'
        elif valor < 2.0:
            return '2. Insuficiente (1-2)'
        elif valor < 4.0:
            return '3. Adequado (2-4)'
        elif valor < 8.0:
            return '4. Bom (4-8)'
        else:
            return '5. Excelente (≥8)'
    
    gdf_mapa_mun['categoria'] = gdf_mapa_mun[col_densidade].apply(classificar_densidade)
    
    # ============================================================
    # PALETA ColorBrewer YlOrRd (amarelo → laranja → vermelho)
    # ============================================================
    # Semântica: mais escuro = MAIOR densidade médica
    # (consistente com heat map de intensidade)
    cores_categorias = {
        '1. Crítico (<1,0)':     '#ffeda0',   # Amarelo claro
        '2. Insuficiente (1-2)': '#feb24c',   # Amarelo-laranja
        '3. Adequado (2-4)':     '#fd8d3c',   # Laranja
        '4. Bom (4-8)':          '#fc4e2a',   # Laranja-vermelho
        '5. Excelente (≥8)':     '#bd0026',   # Vermelho escuro
    }
    
    #TODO: Corrigir Layer com nomes dos municípios na camada mais acima
    # 5. Criar mapa base
    centro_sp = [-22.19, -48.71]
    m1 = folium.Map(location=centro_sp, zoom_start=7, tiles='CartoDB positron')
    
    # 6. Adicionar camadas por categoria
    for categoria, cor in cores_categorias.items():
        gdf_cat = gdf_mapa_mun[gdf_mapa_mun['categoria'] == categoria].copy()
        
        if len(gdf_cat) > 0:
            geo_json = gdf_cat.to_json()
            
            folium.GeoJson(
                geo_json,
                name=categoria,
                style_function=lambda feature, cor=cor: {
                    'fillColor': cor,
                    'color': 'white',          
                    'weight': 0.8,
                    'fillOpacity': 0.65,       
                                               
                },
                highlight_function=lambda feature: {
                    'weight': 2.5,
                    'color': '#333',
                    'fillOpacity': 0.85        
                },
                tooltip=folium.GeoJsonTooltip(
                    fields=[col_nm_mun, col_pop, col_medicos, col_densidade],
                    aliases=['🏙️ Município:', ' População:', '🩺 Médicos SUS:', '📊 Densidade (por 1k hab):'],
                    style=('background-color: white; color: #333333; font-family: arial; '
                           'font-size: 12px; padding: 10px; border-radius: 5px;'),
                    localize=True,
                    sticky=True
                )
            ).add_to(m1)
    
    # 7. Legenda customizada (YlOrRd)
    legenda_html = '''
    <div style="position: fixed; 
                bottom: 30px; left: 30px; 
                width: 260px; 
                background-color: rgba(255,255,255,0.95); 
                border: 2px solid #333; 
                border-radius: 8px;
                padding: 14px;
                font-size: 13px;
                z-index: 9999;
                box-shadow: 3px 3px 8px rgba(0,0,0,0.3);">
        <b style="font-size:14px;">🩺 Densidade Médica SUS</b><br>
        <small>(médicos por 1.000 hab)</small><br>
        <small style="color:#666;">Mais escuro = mais médicos</small><br><br>
        <i style="background:#bd0026;width:18px;height:18px;float:left;margin-right:8px;opacity:0.8;border-radius:3px;"></i>
        Excelente (≥8)<br>
        <i style="background:#fc4e2a;width:18px;height:18px;float:left;margin-right:8px;opacity:0.8;border-radius:3px;"></i>
        Bom (4-8)<br>
        <i style="background:#fd8d3c;width:18px;height:18px;float:left;margin-right:8px;opacity:0.8;border-radius:3px;"></i>
        Adequado (2-4)<br>
        <i style="background:#feb24c;width:18px;height:18px;float:left;margin-right:8px;opacity:0.8;border-radius:3px;"></i>
        Insuficiente (1-2)<br>
        <i style="background:#ffeda0;width:18px;height:18px;float:left;margin-right:8px;opacity:0.8;border-radius:3px;"></i>
        Crítico (&lt;1,0)
    </div>
    '''
    m1.get_root().html.add_child(folium.Element(legenda_html))
    
    # 8. Título
    titulo_html = '''
    <div style="position: fixed; 
                top: 10px; left: 50%; 
                transform: translateX(-50%);
                background-color: rgba(255,255,255,0.95); 
                border: 2px solid #333; 
                border-radius: 8px;
                padding: 10px 20px;
                font-size: 16px;
                font-weight: bold;
                z-index: 9999;
                box-shadow: 3px 3px 8px rgba(0,0,0,0.3);">
         Densidade Médica SUS - Estado de São Paulo
    </div>
    '''
    m1.get_root().html.add_child(folium.Element(titulo_html))
    
    # 9. Controle de camadas
    folium.LayerControl(collapsed=False).add_to(m1)
    
    # 10. Salvar
    output_map = OUTPUT_MAPAS / 'mapa_01_densidade_municipal.html'
    m1.save(output_map)
    
    print(f"\n💾 Mapa 1 salvo em: {output_map}")
    print("🌐 Abra o arquivo HTML no navegador!")
    
    # Estatísticas
    print("\n Distribuição por categoria:")
    display(
        gdf_mapa_mun.groupby('categoria')
        .agg(
            municipios=(col_nm_mun, 'count'),
            populacao_total=(col_pop, 'sum'),
            medicos_total=(col_medicos, 'sum')
        )
        .assign(
            pct_pop=lambda x: (x['populacao_total'] / x['populacao_total'].sum() * 100).round(2),
            densidade_media=lambda x: (x['medicos_total'] / x['populacao_total'] * 1000).round(2)
        )
    )
    
    display(m1)

In [ ]:
# ============================================================
# MAPA 2: Densidade Populacional - Um mapa por município
# ============================================================


print("=" * 70)
print("GERANDO MAPAS DE DENSIDADE POPULACIONAL POR MUNICÍPIO")
print("=" * 70)

if gdf_setores_geo is None:
    print(" Shapefile de setores não disponível - pulando")
else:
    # ============================================================
    # 1. Preparar dados
    # ============================================================
    print("\n Preparando dados...")
    
    colunas_df_setores = df_setores.columns.tolist()
    col_pop_df = next((c for c in colunas_df_setores if c.lower() in ['v0001', 'populacao', 'população']), None)
    
    
    cols_para_merge = ['CD_SETOR', 'dist_minima_metros', 'categoria_acesso', 'acessibilidade_e2sfca', 'total_medicos_dentro']
    if col_pop_df:
        cols_para_merge.append(col_pop_df)
    
    gdf_mapa_setores = gdf_setores_geo.merge(
        df_setores[cols_para_merge],
        on='CD_SETOR',
        how='left',
        suffixes=('', '_dados')
    )
    
    # Limpar colunas duplicadas
    cols_para_remover = [c for c in gdf_mapa_setores.columns if c.endswith('_dados')]
    if cols_para_remover:
        gdf_mapa_setores = gdf_mapa_setores.drop(columns=cols_para_remover)
    
    # Normalizar nome da população
    if col_pop_df and col_pop_df != 'v0001':
        gdf_mapa_setores = gdf_mapa_setores.rename(columns={col_pop_df: 'v0001'})
    
    # Detectar coluna de município
    colunas = gdf_mapa_setores.columns.tolist()
    col_nm_mun = next((c for c in colunas if c == 'NM_MUN' or c.startswith('NM_MUN')), None)
    
    # Tratar nulos
    gdf_mapa_setores['v0001'] = gdf_mapa_setores['v0001'].fillna(0).astype(int)
    gdf_mapa_setores['area_final_km2'] = gdf_mapa_setores['AREA_KM2'].fillna(0.001)
    gdf_mapa_setores.loc[gdf_mapa_setores['area_final_km2'] < 0.001, 'area_final_km2'] = 0.001
    
    # Calcular densidade
    gdf_mapa_setores['densidade_pop'] = (
        gdf_mapa_setores['v0001'] / gdf_mapa_setores['area_final_km2']
    ).round(2)
    
    # ============================================================
    # 2. Converter para WGS84 (EPSG:4326)
    # ============================================================
    print("\n Convertendo para WGS84 (EPSG:4326)...")
    gdf_mapa_setores = gdf_mapa_setores.to_crs('EPSG:4326')
    print(f"    CRS: {gdf_mapa_setores.crs}")
    print(f"    {len(gdf_mapa_setores):,} setores processados")
    
    # ============================================================
    # 3. Lista de municípios
    # ============================================================
    municipios_unicos = sorted(gdf_mapa_setores[col_nm_mun].dropna().unique().tolist())
    print(f"    {len(municipios_unicos)} municípios")
    
    # ============================================================
    # 4. Criar página índice
    # ============================================================
    print("\n Criando página índice...")
    
    indice_html = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Mapas de Densidade Populacional - SP</title>
        <meta charset="UTF-8">
        <style>
            body { font-family: Arial, sans-serif; padding: 20px; background: #f5f5f5; }
            h1 { color: #333; text-align: center; }
            .search-box { 
                width: 100%; padding: 12px; font-size: 16px; margin: 20px 0;
                border: 2px solid #ddd; border-radius: 8px;
            }
            .municipios { 
                display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 10px; margin-top: 20px;
            }
            .municipio { 
                background: white; padding: 12px; border-radius: 6px; border: 1px solid #ddd; transition: all 0.3s;
            }
            .municipio:hover { 
                background: #fff5e6; transform: translateY(-2px); box-shadow: 0 2px 8px rgba(0,0,0,0.1);
            }
            .municipio a { text-decoration: none; color: #d95f0e; font-weight: bold; }
            .municipio small { display: block; color: #666; margin-top: 4px; font-size: 12px; }
        </style>
    </head>
    <body>
        <h1>🗺️ Mapas de Densidade Populacional - São Paulo</h1>
        <input type="text" class="search-box" placeholder="🔍 Buscar município..." id="searchInput">
        <div class="municipios" id="municipiosList">
    """
    
    # ============================================================
    # 5. Gera um mapa HTML para cada município
    # ============================================================
    print("\n Gerando mapas individuais...")

    # Paleta ColorBrewer YlOrRd (9 classes)
    PALETA_YLORRD = {
        0:         '#ffffcc',   # Sem população / muito baixa
        100:       '#ffeda0',   # < 100 hab/km²
        500:       '#fed976',   # 100 - 500
        1000:      '#feb24c',   # 500 - 1.000
        5000:      '#fd8d3c',   # 1.000 - 5.000
        10000:     '#fc4e2a',   # 5.000 - 10.000
        20000:     '#e31a1c',   # 10.000 - 20.000
        50000:     '#bd0026',   # 20.000 - 50.000
        float('inf'): '#800026'  # >= 50.000
    }

    def cor_para_densidade(densidade):
        """Retorna a cor da paleta YlOrRd para uma dada densidade."""
        if densidade == 0 or pd.isna(densidade):
            return '#f0f0f0'  # Cinza claro para sem população
        for limite, cor in PALETA_YLORRD.items():
            if densidade < limite:
                return cor
        return PALETA_YLORRD[float('inf')]

    for i, municipio in enumerate(municipios_unicos, 1):
        gdf_mun = gdf_mapa_setores[gdf_mapa_setores[col_nm_mun] == municipio].copy()
        
        if len(gdf_mun) == 0:
            continue
        
        # Calcular bounds do município
        bounds = gdf_mun.total_bounds
        centro_lat = (bounds[1] + bounds[3]) / 2
        centro_lon = (bounds[0] + bounds[2]) / 2
        
        # Calcular zoom baseado no tamanho do bounding box
        lat_diff = abs(bounds[3] - bounds[1])
        lon_diff = abs(bounds[2] - bounds[0])
        tamanho_mun = max(lat_diff, lon_diff)
        
        if tamanho_mun > 0.5:
            zoom_inicial = 11
        elif tamanho_mun > 0.2:
            zoom_inicial = 12
        elif tamanho_mun > 0.05:
            zoom_inicial = 13
        else:
            zoom_inicial = 14
        
        # Criar mapa
        m = folium.Map(location=[centro_lat, centro_lon], zoom_start=zoom_inicial, tiles='CartoDB voyager no labels')
        folium.map.CustomPane("linhas_base_topo", z_index=640).add_to(m)
        folium.map.CustomPane("labels_topo", z_index=650).add_to(m)
        
        FILL_OPACITY_SETORES = 0.85
        LINHAS_OPACITY = 0.40
        PADDING_FIT_BOUNDS = [5, 5]

        def style_function(feature):
            densidade = feature['properties'].get('densidade_pop', 0)
            cor = cor_para_densidade(densidade)
            return {
                'fillColor': cor,
                'fillOpacity': FILL_OPACITY_SETORES,
                'color': '#ffffff',
                'weight': 0.8,
                'opacity': 0.6
            }
        
        geojson_mun = gdf_mun.__geo_interface__
        
        geojson_layer = folium.GeoJson(
            geojson_mun,
            style_function=style_function,
            tooltip=folium.GeoJsonTooltip(
                fields=[col_nm_mun, 'CD_SETOR', 'v0001', 'area_final_km2', 'densidade_pop', 'categoria_acesso', 'acessibilidade_e2sfca'],
                aliases=['🏙️ Município:', '📍 Setor:', '👥 População:', '📐 Área (km²):', '📊 Densidade (hab/km²):', '🏥 Acesso (E2SFCA):', '📈 Índice E2SFCA:'],
                style='background-color: white; color: #333; font-family: arial; font-size: 12px; padding: 10px; border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);',
                localize=True
            ),
            highlight_function=lambda feature: {
                'fillOpacity': 0.95,      
                'weight': 2.5,
                'color': '#000000'       
            }
        ).add_to(m)

        geojson_layer.get_root().header.add_child(folium.Element("""
            <style>
                .leaflet-tooltip-pane { z-index: 9999 !important; }
            </style>
        """))
        
        folium.TileLayer(
            tiles='https://{s}.basemaps.cartocdn.com/rastertiles/voyager_nolabels/{z}/{x}/{y}{r}.png',
            attr='© OpenStreetMap contributors © CARTO',
            name='Linhas das Ruas e Quadras',
            overlay=True, control=False, opacity=LINHAS_OPACITY, pane="linhas_base_topo"
        ).add_to(m)

        folium.TileLayer(
            tiles='CartoDB voyager only labels',
            name='Nomes dos Bairros e Ruas',
            overlay=True, control=False, pane="labels_topo"
        ).add_to(m)

        m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]], padding=PADDING_FIT_BOUNDS)
        
        # ============================================================
        # LEGENDA (YlOrRd)
        # ============================================================
        legenda = '''
        <div style="position: fixed; bottom: 10px; left: 10px; background: rgba(255,255,255,0.95); 
                    padding: 12px; border: 2px solid #333; border-radius: 6px; font-size: 11px; z-index: 1000;
                    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
            <b style="font-size:13px;">📊 Densidade Populacional</b><br>
            <small>(habitantes por km²)</small><br><br>
            <span style="color:#800026">■</span> ≥50.000<br>
            <span style="color:#bd0026">■</span> 20.000 - 50.000<br>
            <span style="color:#e31a1c">■</span> 10.000 - 20.000<br>
            <span style="color:#fc4e2a">■</span> 5.000 - 10.000<br>
            <span style="color:#fd8d3c">■</span> 1.000 - 5.000<br>
            <span style="color:#feb24c">■</span> 500 - 1.000<br>
            <span style="color:#fed976">■</span> 100 - 500<br>
            <span style="color:#ffeda0">■</span> <100<br>
            <span style="color:#f0f0f0">■</span> Sem população
        </div>
        '''
        m.get_root().html.add_child(folium.Element(legenda))
        
        # Título no topo do mapa
        titulo = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
                    background-color: rgba(255,255,255,0.95); border: 2px solid #333; 
                    border-radius: 8px; padding: 10px 20px; font-size: 14px; font-weight: bold;
                    z-index: 1000; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
            📊 Densidade Populacional - {municipio}
        </div>
        '''
        m.get_root().html.add_child(folium.Element(titulo))
        
        # Salvar
        nome_arquivo = municipio.replace(' ', '_').replace('/', '_').lower()
        output_file = OUTPUT_MAPAS / f'mapa_{nome_arquivo}.html'
        m.save(output_file)
        
        # Adicionar à página índice
        indice_html += f"""
        <div class="municipio">
            <a href="mapa_{nome_arquivo}.html">{municipio}</a>
            <small>{len(gdf_mun)} setores | {gdf_mun['v0001'].sum():,} hab</small>
        </div>
        """
        
        if i % 50 == 0:
            print(f"   Progresso: {i}/{len(municipios_unicos)} municípios...")
    
    # ============================================================
    # 6. Fechar página índice
    # ============================================================
    indice_html += """
        </div>
        <script>
        document.getElementById('searchInput').addEventListener('input', function(e) {
            var filter = e.target.value.toLowerCase();
            var municipios = document.querySelectorAll('.municipio');
            municipios.forEach(function(m) {
                var text = m.textContent.toLowerCase();
                m.style.display = text.indexOf(filter) > -1 ? 'block' : 'none';
            });
        });
        </script>
    </body>
    </html>
    """
    
    # Salvar índice
    indice_file = OUTPUT_MAPAS / 'indice_mapas.html'
    with open(indice_file, 'w', encoding='utf-8') as f:
        f.write(indice_html)
    
    print(f"\n {len(municipios_unicos)} mapas gerados!")
    print(f" Página índice: {indice_file}")
    print("\n Como usar:")
    print("   1. Abra 'indice_mapas.html' no navegador")
    print("   2. Use a busca para encontrar um município")
    print("   3. Clique no link para ver o mapa detalhado")
    print("   4. Passe o mouse sobre os setores para ver a densidade E a acessibilidade E2SFCA!")

In [ ]:
# ============================================================
# MAPA 3: Bivariado Municipal - Densidade Médica × Acessibilidade (E2SFCA)
# ============================================================
# Versão MUNICIPAL: ~645 polígonos
# Combina: Densidade médica (médicos/1k hab) × Acessibilidade (mediana do índice E2SFCA)

print("=" * 70)
print("GERANDO MAPA 3: BIVARIADO MUNICIPAL (E2SFCA)")
print("=" * 70)

if gdf_municipios is None or df_comparacao is None:
    print(" Dados municipais não disponíveis - pulando")
else:
    # ============================================================
    # 1. Preparar dados
    # ============================================================
    print("\n Preparando dados...")
    
    df_bivariado = df_comparacao.copy()
    
    # Garantir coluna de merge
    df_bivariado['cod_mun_6'] = df_bivariado['cod_mun_ibge'].astype(str).str[:6]
    
    col_nm_mun = 'nm_mun'
    col_densidade = 'medicos_por_1k'
    
    col_acesso_e2sfca = 'acessibilidade_e2sfca_mediana'
    
    # Fallback caso a coluna não exista (segurança)
    if col_acesso_e2sfca not in df_bivariado.columns:
        print(f"    Coluna '{col_acesso_e2sfca}' não encontrada. Usando 'dist_mediana_km' como fallback.")
        col_acesso_e2sfca = 'dist_mediana_km'
        usar_distancia = True
    else:
        usar_distancia = False
    
    print(f"    {len(df_bivariado):,} municípios")
    
    # ============================================================
    # 2. Classificar em categorias 3×3
    # ============================================================
    print("\n Classificando municípios...")
    
    # Classificar densidade médica (tercis)
    densidade_quartis = df_bivariado[col_densidade].quantile([0.33, 0.66])
    
    def classificar_densidade(valor):
        if pd.isna(valor) or valor == 0:
            return 'Baixa'
        elif valor < densidade_quartis[0.33]:
            return 'Baixa'
        elif valor < densidade_quartis[0.66]:
            return 'Média'
        else:
            return 'Alta'
    
    df_bivariado['cat_densidade'] = df_bivariado[col_densidade].apply(classificar_densidade)
    
    # Classificar acessibilidade
    if not usar_distancia:
        acesso_quartis = df_bivariado[col_acesso_e2sfca].quantile([0.33, 0.66])
        
        def classificar_acesso(valor):
            if pd.isna(valor) or valor == 0:
                return 'Baixo'
            elif valor < acesso_quartis[0.33]:
                return 'Baixo'   # Índice baixo = mau acesso
            elif valor < acesso_quartis[0.66]:
                return 'Médio'
            else:
                return 'Alto'    # Índice alto = bom acesso
        
        alias_acesso = ' Índice E2SFCA mediano:'
    else:
        # Lógica legada (distância: valores MENORES = MELHOR acesso)
        distancia_quartis = df_bivariado[col_acesso_e2sfca].quantile([0.33, 0.66])
        
        def classificar_acesso(valor):
            if pd.isna(valor):
                return 'Baixo'
            elif valor < distancia_quartis[0.33]:
                return 'Alto'    # Perto = bom acesso
            elif valor < distancia_quartis[0.66]:
                return 'Médio'
            else:
                return 'Baixo'   # Longe = mau acesso
        
        alias_acesso = ' Dist. mediana ao médico (km):'
    
    df_bivariado['cat_acesso'] = df_bivariado[col_acesso_e2sfca].apply(classificar_acesso)
    
    # Categoria bivariada combinada
    df_bivariado['categoria_bivariada'] = (
        df_bivariado['cat_densidade'] + ' + ' + df_bivariado['cat_acesso']
    )
    
    print(f"   Classificação concluída")
    
    # ============================================================
    # 3. Merge com shapefile
    # ============================================================
    print("\n Merge com shapefile de municípios...")
    
    gdf_bivariado = gdf_municipios.merge(
        df_bivariado[['cod_mun_6', 'categoria_bivariada', 'cat_densidade', 
                      'cat_acesso', col_densidade, col_acesso_e2sfca, col_nm_mun]],
        on='cod_mun_6',
        how='left'
    )
    
    # Preencher eventuais nulos com 'Sem dados'
    gdf_bivariado['categoria_bivariada'] = gdf_bivariado['categoria_bivariada'].fillna('Sem dados')
    
    print(f"    {len(gdf_bivariado):,} municípios no mapa")
    
    # ============================================================
    # 4. Paleta de cores bivariada (matriz 3×3)
    # ============================================================
    paleta_bivariada = {
        # Densidade Alta
        'Alta + Baixo':  '#800026',   # 🔴 CRÍTICO: Médicos no município, mas acesso real baixo (desigualdade)
        'Alta + Médio':  '#fc4e2a',   # 🟠 ATENÇÃO
        'Alta + Alto':   '#41ab5d',   # 🟢 BOM: Muitos médicos, bom acesso
        
        # Densidade Média
        'Média + Baixo': '#e34a33',   # 🟠 ATENÇÃO
        'Média + Médio': '#fed976',   # 🟡 MODERADO
        'Média + Alto':  '#a1d99b',   # 🟢 BOM
        
        # Densidade Baixa
        'Baixa + Baixo': '#969696',   # ⚫ DESERTO RURAL: Poucos médicos E acesso ruim
        'Baixa + Médio': '#d9d9d9',   # ⚪ RURAL: Pouca densidade, acesso moderado
        'Baixa + Alto':  '#c6dbef',   # 🔵 RARO: Pouca densidade, mas bom acesso
        
        'Sem dados':     '#f0f0f0',
    }
    
    # ============================================================
    # 5. Criar mapa
    # ============================================================
    print("\n Criando mapa...")
    
    centro_sp = [-22.19, -48.71]
    m3 = folium.Map(location=centro_sp, zoom_start=7, tiles='CartoDB positron')
    
    def style_function(feature):
        categoria = feature['properties'].get('categoria_bivariada', 'Sem dados')
        cor = paleta_bivariada.get(categoria, '#f0f0f0')
        
        return {
            'fillColor': cor,
            'color': 'white',
            'weight': 0.8,
            'fillOpacity': 0.75,
        }
    
    folium.GeoJson(
        gdf_bivariado,
        style_function=style_function,
        tooltip=folium.GeoJsonTooltip(
            fields=[col_nm_mun, col_densidade, col_acesso_e2sfca, 'cat_densidade', 'cat_acesso'],
            aliases=['🏙️ Município:', '🩺 Densidade (méd/1k hab):', 
                     alias_acesso,
                     '📈 Densidade:', '🏥 Acesso:'],
            style='background-color: white; color: #333; font-family: arial; font-size: 12px; padding: 10px; border-radius: 5px;',
            localize=True
        ),
        highlight_function=lambda feature: {
            'fillOpacity': 0.95,
            'weight': 2.5,
            'color': '#000000'
        }
    ).add_to(m3)
    
    # ============================================================
    # 6. Legenda bivariada (matriz 3×3)
    # ============================================================
    print("\n Adicionando legenda...")
    
    legenda_html = '''
    <div style="position: fixed; 
                bottom: 30px; left: 30px; 
                width: 340px; 
                background-color: rgba(255,255,255,0.98); 
                border: 2px solid #333; 
                border-radius: 8px;
                padding: 15px;
                font-size: 11px;
                z-index: 9999;
                box-shadow: 3px 3px 8px rgba(0,0,0,0.3);">
        <b style="font-size:13px;"> Mapa Bivariado: Densidade Médica x Acessibilidade (E2SFCA)</b><br>
        <small style="color:#666;">Nível municipal | Identificação de desertos médicos</small><br><br>
        
        <table style="width:100%; border-collapse: collapse;">
            <tr>
                <td style="padding:2px;"></td>
                <td style="text-align:center; font-weight:bold; padding:2px;">Baixo Acesso<br><small>(índice baixo)</small></td>
                <td style="text-align:center; font-weight:bold; padding:2px;">Médio Acesso</td>
                <td style="text-align:center; font-weight:bold; padding:2px;">Alto Acesso<br><small>(índice alto)</small></td>
            </tr>
            <tr>
                <td style="font-weight:bold; padding:2px;">Alta Densidade<br><small>(médicos/1k)</small></td>
                <td style="background:#800026; width:70px; height:35px; border:1px solid #333;"></td>
                <td style="background:#fc4e2a; width:70px; height:35px; border:1px solid #333;"></td>
                <td style="background:#41ab5d; width:70px; height:35px; border:1px solid #333;"></td>
            </tr>
            <tr>
                <td style="font-weight:bold; padding:2px;">Média Densidade</td>
                <td style="background:#e34a33; width:70px; height:35px; border:1px solid #333;"></td>
                <td style="background:#fed976; width:70px; height:35px; border:1px solid #333;"></td>
                <td style="background:#a1d99b; width:70px; height:35px; border:1px solid #333;"></td>
            </tr>
            <tr>
                <td style="font-weight:bold; padding:2px;">Baixa Densidade</td>
                <td style="background:#969696; width:70px; height:35px; border:1px solid #333;"></td>
                <td style="background:#d9d9d9; width:70px; height:35px; border:1px solid #333;"></td>
                <td style="background:#c6dbef; width:70px; height:35px; border:1px solid #333;"></td>
            </tr>
        </table>
        
        <br>
        <small>
            <b>🔴 Vermelho escuro</b> = CRÍTICO (desigualdade: médicos no município, mas acesso real baixo)<br>
            <b>⚫ Cinza escuro</b> = DESERTO RURAL (poucos médicos E acesso ruim)<br>
            <b>🟢 Verde</b> = BOM (bom acesso a médicos)<br>
            <small style="color:#888;">(Cores cinzas representam municípios de baixa densidade populacional)</small>
        </small>
    </div>
    '''
    m3.get_root().html.add_child(folium.Element(legenda_html))
    
    # Título
    titulo_html = '''
    <div style="position: fixed; 
                top: 10px; left: 50%; 
                transform: translateX(-50%);
                background-color: rgba(255,255,255,0.95); 
                border: 2px solid #333; 
                border-radius: 8px;
                padding: 12px 24px;
                font-size: 16px;
                font-weight: bold;
                z-index: 9999;
                box-shadow: 3px 3px 8px rgba(0,0,0,0.3);">
         Desertos Médicos em SP: Densidade x Acessibilidade (Municipal - E2SFCA)
    </div>
    '''
    m3.get_root().html.add_child(folium.Element(titulo_html))
    
    # Salvar
    output_map = OUTPUT_MAPAS / 'mapa_03_bivariado_municipal.html'
    m3.save(output_map)
    
    print(f"\n Mapa 3 salvo em: {output_map}")
    print(" Abra o arquivo HTML no navegador!")
    
    # ============================================================
    # 7. Estatísticas Finais
    # ============================================================
    print("\n Análise de Desertos Médicos:")
    
    municipios_criticos = gdf_bivariado[gdf_bivariado['categoria_bivariada'] == 'Baixa + Baixo']
    print(f"   ⚫ Municípios DESERTO RURAL (baixa densidade, baixo acesso): {len(municipios_criticos):,}")
    
    municipios_paradoxo = gdf_bivariado[gdf_bivariado['categoria_bivariada'] == 'Alta + Baixo']
    print(f"   🔴 Municípios CRÍTICOS/PARADOXAIS (alta densidade, baixo acesso): {len(municipios_paradoxo):,}")
    print(f"      → Indica desigualdade intra-municipal (médicos concentrados no centro, periferia com acesso real baixo)")
    
    display(m3)

In [ ]:
# ============================================================
# MAPA 4: Bivariado Setorial - Densidade Pop × Acesso a Médicos (E2SFCA)
# ============================================================


print("=" * 70)
print("GERANDO MAPA 4: BIVARIADO SETORIAL POR MUNICÍPIO")
print("=" * 70)

if gdf_setores_geo is None:
    print(" Shapefile de setores não disponível - pulando")
else:
    # ============================================================
    # 1. Preparar dados
    # ============================================================
    print("\n Preparando dados...")

    colunas_df_setores = df_setores.columns.tolist()

    col_pop_df = next(
        (c for c in colunas_df_setores if c.lower() in ['v0001', 'populacao', 'população']),
        None
    )
    if col_pop_df is None:
        print("    ATENÇÃO: coluna de população não encontrada em df_setores!")
        print(f"   Colunas disponíveis: {colunas_df_setores}")
    else:
        print(f"   Coluna de população detectada: '{col_pop_df}'")

    cols_para_merge = ['CD_SETOR', 'dist_minima_metros', 'categoria_acesso', 'total_medicos_dentro']
    if col_pop_df:
        cols_para_merge.append(col_pop_df)

    n_dup_chave = df_setores['CD_SETOR'].duplicated().sum()
    if n_dup_chave > 0:
        print(f"    ATENÇÃO: {n_dup_chave} CD_SETOR duplicados em df_setores")

    n_linhas_antes = len(gdf_setores_geo)

    gdf_mapa_setores = gdf_setores_geo.merge(
        df_setores[cols_para_merge],
        on='CD_SETOR',
        how='left',
        suffixes=('', '_dados')
    )

    if len(gdf_mapa_setores) != n_linhas_antes:
        print(f"    ATENÇÃO: merge alterou contagem de linhas ({n_linhas_antes:,} → {len(gdf_mapa_setores):,})")
    else:
        print(f"    Merge preservou {n_linhas_antes:,} linhas")

    # Limpar colunas duplicadas
    cols_para_remover = [c for c in gdf_mapa_setores.columns if c.endswith('_dados')]
    if cols_para_remover:
        gdf_mapa_setores = gdf_mapa_setores.drop(columns=cols_para_remover)

    # Normalizar nome da população
    if col_pop_df and col_pop_df != 'v0001':
        gdf_mapa_setores = gdf_mapa_setores.rename(columns={col_pop_df: 'v0001'})

    # Detectar coluna de município
    colunas = gdf_mapa_setores.columns.tolist()
    col_nm_mun = next((c for c in colunas if c == 'NM_MUN' or c.startswith('NM_MUN')), None)
    if col_nm_mun is None:
        raise ValueError("Coluna de município (NM_MUN) não encontrada — abortando Mapa 4")

    # Tratar nulos
    if 'v0001' not in gdf_mapa_setores.columns:
        gdf_mapa_setores['v0001'] = 0
    else:
        gdf_mapa_setores['v0001'] = gdf_mapa_setores['v0001'].fillna(0).astype(int)
        
    gdf_mapa_setores['area_final_km2'] = gdf_mapa_setores['AREA_KM2'].fillna(0.001)
    gdf_mapa_setores.loc[gdf_mapa_setores['area_final_km2'] < 0.001, 'area_final_km2'] = 0.001

    gdf_mapa_setores['densidade_pop'] = (
        gdf_mapa_setores['v0001'] / gdf_mapa_setores['area_final_km2']
    ).round(2)

    print(f"    {len(gdf_mapa_setores):,} setores processados")
    
    # ============================================================
    # 2. Classificar em categorias bivariadas (3×3 + Irrelevante)
    # ============================================================
    print("\n Classificando setores em categorias bivariadas...")
    
    # Definir threshold de população mínima para evitar ruído visual
    POPULACAO_MINIMA = 50
    print(f"    Threshold de população mínima: {POPULACAO_MINIMA} hab")
    
    gdf_mapa_setores['populacao_relevante'] = (gdf_mapa_setores['v0001'] >= POPULACAO_MINIMA)
    n_irrelevantes = (~gdf_mapa_setores['populacao_relevante']).sum()
    print(f"    {n_irrelevantes:,} setores com população < {POPULACAO_MINIMA} (serão classificados como 'Irrelevante')")

    print("    Classificando densidade populacional por município (apenas setores relevantes)...")

    def classificar_densidade_por_municipio(grupo):
        """Classifica densidade em tercis dentro de cada município."""
        densidade = grupo['densidade_pop']

        if len(grupo) < 10:
            return pd.cut(densidade, bins=[0, 100, 1000, float('inf')], labels=['Baixa', 'Média', 'Alta'], include_lowest=True)

        try:
            return pd.qcut(densidade, q=[0, 0.33, 0.66, 1.0], labels=['Baixa', 'Média', 'Alta'], duplicates='drop')
        except ValueError:
            return pd.cut(densidade, bins=[0, 100, 1000, float('inf')], labels=['Baixa', 'Média', 'Alta'], include_lowest=True)

    # Aplicar apenas aos setores com população relevante
    gdf_mapa_setores.loc[gdf_mapa_setores['populacao_relevante'], 'cat_densidade_pop'] = (
        gdf_mapa_setores[gdf_mapa_setores['populacao_relevante']]
        .groupby(col_nm_mun, group_keys=False)
        .apply(classificar_densidade_por_municipio, include_groups=False)
    )

    
    gdf_mapa_setores['cat_densidade_pop'] = gdf_mapa_setores['cat_densidade_pop'].astype(str)

    # Setores irrelevantes recebem categoria especial
    gdf_mapa_setores.loc[~gdf_mapa_setores['populacao_relevante'], 'cat_densidade_pop'] = 'Irrelevante'

    print("\n   Valores únicos em 'categoria_acesso' (do E2SFCA):")
    print(gdf_mapa_setores['categoria_acesso'].value_counts(dropna=False))

    def classificar_acesso_medicos(categoria):
        """Classifica acesso E2SFCA em Alto/Médio/Baixo para mapa bivariado."""
        if pd.isna(categoria):
            return 'Baixo'
        
        cat_str = str(categoria).strip()
        
        # Alto acesso: Excelente ou Bom
        if 'Excelente' in cat_str or 'Bom' in cat_str:
            return 'Alto'
        # Médio acesso: Moderado
        elif 'Moderado' in cat_str:
            return 'Médio'
        # Baixo acesso: Limitado, Crítico ou Deserto
        else:
            return 'Baixo'

    gdf_mapa_setores['cat_acesso_medicos'] = gdf_mapa_setores['categoria_acesso'].apply(
        classificar_acesso_medicos
    )

    pct_medio = (gdf_mapa_setores['cat_acesso_medicos'] == 'Médio').mean() * 100
    print(f"    {pct_medio:.1f}% dos setores classificados como 'Médio'")

    # Criar categoria bivariada combinada
    # Setores irrelevantes → categoria especial
    gdf_mapa_setores.loc[~gdf_mapa_setores['populacao_relevante'], 'categoria_bivariada'] = 'Irrelevante'

    # Setores relevantes → combinação normal
    gdf_mapa_setores.loc[gdf_mapa_setores['populacao_relevante'], 'categoria_bivariada'] = (
        gdf_mapa_setores.loc[gdf_mapa_setores['populacao_relevante'], 'cat_densidade_pop'] + 
        ' + ' + 
        gdf_mapa_setores.loc[gdf_mapa_setores['populacao_relevante'], 'cat_acesso_medicos']
    )

    print(f"    Classificação concluída")

    # ============================================================
    # 3. Paleta de cores bivariada (matriz 3×3 + Irrelevante)
    # ============================================================
    paleta_bivariada = {
        # Densidade Alta
        'Alta + Baixo':  '#800026',   # 🔴 CRÍTICO
        'Alta + Médio':  '#e34a33',   # 🟠 ATENÇÃO
        'Alta + Alto':   '#41ab5d',   # 🟢 BOM
        # Densidade Média
        'Média + Baixo': '#fb6a4a',   # 🟠 ATENÇÃO
        'Média + Médio': '#fed976',   # 🟡 MODERADO
        'Média + Alto':  '#a1d99b',   # 🟢 BOM
        # Densidade Baixa
        'Baixa + Baixo': '#969696',   # ⚫ DESERTO
        'Baixa + Médio': '#d9d9d9',   # ⚪ RURAL
        'Baixa + Alto':  '#c6dbef',   # 🔵 RARO
        # Nova categoria
        'Irrelevante':   '#f0f0f0',   # 🌫️ CINZA MUITO CLARO (população < 100 hab)
        'Sem dados':     '#ffffff',
    }

    categorias_nao_mapeadas = ~gdf_mapa_setores['categoria_bivariada'].isin(paleta_bivariada.keys())
    n_nao_mapeados = categorias_nao_mapeadas.sum()
    if n_nao_mapeados > 0:
        print(f"\n    ATENÇÃO: {n_nao_mapeados:,} setores com categoria fora da paleta:")
        print(gdf_mapa_setores.loc[categorias_nao_mapeadas, 'categoria_bivariada'].value_counts())
    else:
        print(f"\n    Todos os setores mapeados corretamente na paleta")
    
    # ============================================================
    # 4. Converter para WGS84
    # ============================================================
    print("\n Convertendo para WGS84 (EPSG:4326)...")
    gdf_mapa_setores = gdf_mapa_setores.to_crs('EPSG:4326')
    print(f"    CRS: {gdf_mapa_setores.crs}")
    
    # ============================================================
    # 5. Lista de municípios
    # ============================================================
    municipios_unicos = sorted(gdf_mapa_setores[col_nm_mun].dropna().unique().tolist())
    print(f"    {len(municipios_unicos)} municípios")
    
    # ============================================================
    # 6. Criar página índice
    # ============================================================
    print("\n Criando página índice...")
    
    indice_html = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Mapas Bivariados - Desertos Médicos por Setor</title>
        <meta charset="UTF-8">
        <style>
            body { font-family: Arial, sans-serif; padding: 20px; background: #f5f5f5; }
            h1 { color: #333; text-align: center; }
            .search-box { width: 100%; padding: 12px; font-size: 16px; margin: 20px 0; border: 2px solid #ddd; border-radius: 8px; }
            .municipios { display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 10px; margin-top: 20px; }
            .municipio { background: white; padding: 12px; border-radius: 6px; border: 1px solid #ddd; transition: all 0.3s; }
            .municipio:hover { background: #fff5e6; transform: translateY(-2px); box-shadow: 0 2px 8px rgba(0,0,0,0.1); }
            .municipio a { text-decoration: none; color: #d95f0e; font-weight: bold; }
            .municipio small { display: block; color: #666; margin-top: 4px; font-size: 12px; }
            .stats { margin-top: 8px; padding-top: 8px; border-top: 1px solid #eee; font-size: 11px; }
            .stats .critico { color: #800026; font-weight: bold; }
        </style>
    </head>
    <body>
        <h1> Desertos Médicos por Setor Censitário</h1>
        <p style="text-align: center; color: #666;">Mapa bivariado: Densidade Populacional × Acesso a Médicos (E2SFCA)</p>
        <input type="text" class="search-box" placeholder="🔍 Buscar município..." id="searchInput">
        <div class="municipios" id="municipiosList">
    """
    
    # ============================================================
    # 7. Gerar um mapa HTML para cada município
    # ============================================================
    print("\n Gerando mapas individuais...")
    
    FILL_OPACITY_SETORES = 0.85
    LINHAS_OPACITY = 0.40
    PADDING_FIT_BOUNDS = [5, 5]

    def style_function(feature):
        categoria = feature['properties'].get('categoria_bivariada', 'Sem dados')
        cor = paleta_bivariada.get(categoria, '#f0f0f0')
        return {
            'fillColor': cor,
            'fillOpacity': FILL_OPACITY_SETORES,
            'color': '#ffffff',
            'weight': 0.8,
            'opacity': 0.6
        }
    
    for i, municipio in enumerate(municipios_unicos, 1):
        gdf_mun = gdf_mapa_setores[gdf_mapa_setores[col_nm_mun] == municipio].copy()
        if len(gdf_mun) == 0:
            continue

        bounds = gdf_mun.total_bounds
        centro_lat = (bounds[1] + bounds[3]) / 2
        centro_lon = (bounds[0] + bounds[2]) / 2

        m = folium.Map(location=[centro_lat, centro_lon], tiles='CartoDB voyager no labels')

        folium.map.CustomPane("linhas_base_topo", z_index=640).add_to(m)
        folium.map.CustomPane("labels_topo", z_index=650).add_to(m)
        
        geojson_mun = gdf_mun.__geo_interface__
        
        geojson_layer = folium.GeoJson(
            geojson_mun,
            style_function=style_function,
            tooltip=folium.GeoJsonTooltip(
                fields=[col_nm_mun, 'CD_SETOR', 'v0001', 'densidade_pop', 'cat_densidade_pop', 'cat_acesso_medicos', 'dist_minima_metros'],
                aliases=['🏙️ Município:', '📍 Setor:', '👥 População:', '📊 Densidade (hab/km²):', '📈 Densidade:', '🏥 Acesso:', '📏 Dist. médico mais próx. (m):'],
                style='background-color: white; color: #333; font-family: arial; font-size: 12px; padding: 10px; border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);',
                localize=True
            ),
            highlight_function=lambda feature: {'fillOpacity': 0.95, 'weight': 2.5, 'color': '#000000'}
        ).add_to(m)
        
        geojson_layer.get_root().header.add_child(folium.Element("<style>.leaflet-tooltip-pane { z-index: 9999 !important; }</style>"))
        
        folium.TileLayer(
            tiles='https://{s}.basemaps.cartocdn.com/rastertiles/voyager_nolabels/{z}/{x}/{y}{r}.png',
            attr='© OpenStreetMap contributors © CARTO',
            name='Linhas das Ruas e Quadras',
            overlay=True, control=False, opacity=LINHAS_OPACITY, pane="linhas_base_topo"
        ).add_to(m)
        
        folium.TileLayer(
            tiles='CartoDB voyager only labels',
            name='Nomes dos Bairros e Ruas',
            overlay=True, control=False, pane="labels_topo"
        ).add_to(m)
        
        m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]], padding=PADDING_FIT_BOUNDS)
        
        # Estatísticas do município
        stats_mun = gdf_mun['categoria_bivariada'].value_counts()
        n_critico = stats_mun.get('Alta + Baixo', 0)
        n_deserto = stats_mun.get('Baixa + Baixo', 0)
        pop_critico = gdf_mun[gdf_mun['categoria_bivariada'] == 'Alta + Baixo']['v0001'].sum()
        
        # Legenda bivariada
        legenda = '''
        <div style="position: fixed; bottom: 10px; left: 10px; background: rgba(255,255,255,0.98); padding: 12px; border: 2px solid #333; border-radius: 6px; font-size: 10px; z-index: 1000; box-shadow: 2px 2px 6px rgba(0,0,0,0.3); max-width: 280px;">
            <b style="font-size:12px;">🗺️ Densidade Pop × Acesso a Médicos</b><br>
            <small style="color:#666;">Identificação de desertos médicos (E2SFCA)</small><br><br>
            <table style="width:100%; border-collapse: collapse;">
                <tr><td style="padding:2px;"></td><td style="text-align:center; font-weight:bold; padding:2px; font-size:9px;">Baixo</td><td style="text-align:center; font-weight:bold; padding:2px; font-size:9px;">Médio</td><td style="text-align:center; font-weight:bold; padding:2px; font-size:9px;">Alto</td></tr>
                <tr><td style="font-weight:bold; padding:2px; font-size:9px;">Alta</td><td style="background:#800026; width:40px; height:20px; border:1px solid #333;"></td><td style="background:#e34a33; width:40px; height:20px; border:1px solid #333;"></td><td style="background:#41ab5d; width:40px; height:20px; border:1px solid #333;"></td></tr>
                <tr><td style="font-weight:bold; padding:2px; font-size:9px;">Média</td><td style="background:#fb6a4a; width:40px; height:20px; border:1px solid #333;"></td><td style="background:#fed976; width:40px; height:20px; border:1px solid #333;"></td><td style="background:#a1d99b; width:40px; height:20px; border:1px solid #333;"></td></tr>
                <tr><td style="font-weight:bold; padding:2px; font-size:9px;">Baixa</td><td style="background:#969696; width:40px; height:20px; border:1px solid #333;"></td><td style="background:#d9d9d9; width:40px; height:20px; border:1px solid #333;"></td><td style="background:#c6dbef; width:40px; height:20px; border:1px solid #333;"></td></tr>
            </table>
            <br><small><b style="color:#800026;">🔴 CRÍTICO</b>: Alta densidade, baixo acesso<br><b style="color:#969696;">⚫ DESERTO</b>: Baixa densidade, baixo acesso<br><b style="color:#41ab5d;">🟢 BOM</b>: Bom acesso a médicos<br><b style="color:#f0f0f0; text-shadow: 1px 1px 1px #999;">⬜ IRRELEVANTE</b>: População < 100 hab</small>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(legenda))
        
        titulo = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); background-color: rgba(255,255,255,0.95); border: 2px solid #333; border-radius: 8px; padding: 10px 20px; font-size: 14px; font-weight: bold; z-index: 1000; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
             Desertos Médicos por Setor - {municipio}
        </div>
        '''
        m.get_root().html.add_child(folium.Element(titulo))
        
        nome_arquivo = municipio.replace(' ', '_').replace('/', '_').lower()
        output_file = OUTPUT_MAPAS / f'mapa_bivariado_{nome_arquivo}.html'
        m.save(output_file)
        
        indice_html += f"""
        <div class="municipio">
            <a href="mapa_bivariado_{nome_arquivo}.html">{municipio}</a>
            <small>{len(gdf_mun)} setores | {gdf_mun['v0001'].sum():,} hab</small>
            <div class="stats">
                <span class="critico">🔴 {n_critico} setores críticos</span> | ⚫ {n_deserto} desertos | 👥 {pop_critico:,} hab em risco
            </div>
        </div>
        """
        
        if i % 50 == 0:
            print(f"   Progresso: {i}/{len(municipios_unicos)} municípios...")
    
    # ============================================================
    # 8. Fechar página índice
    # ============================================================
    indice_html += """
        </div>
        <script>
        document.getElementById('searchInput').addEventListener('input', function(e) {
            var filter = e.target.value.toLowerCase();
            var municipios = document.querySelectorAll('.municipio');
            municipios.forEach(function(m) {
                var text = m.textContent.toLowerCase();
                m.style.display = text.indexOf(filter) > -1 ? 'block' : 'none';
            });
        });
        </script>
    </body>
    </html>
    """
    
    indice_file = OUTPUT_MAPAS / 'indice_mapas_bivariados.html'
    with open(indice_file, 'w', encoding='utf-8') as f:
        f.write(indice_html)
    
    print(f"\n {len(municipios_unicos)} mapas bivariados gerados!")
    print(f" Página índice: {indice_file}")
    
    # ============================================================
    # 9. Estatísticas estaduais
    # ============================================================
    print("\n Estatísticas Estaduais:")
    print(f"   Total de setores: {len(gdf_mapa_setores):,}")
    print(f"   População total: {gdf_mapa_setores['v0001'].sum():,}")
    
    # Setores irrelevantes
    setores_irrelevantes = gdf_mapa_setores[gdf_mapa_setores['categoria_bivariada'] == 'Irrelevante']
    print(f"\n    Setores IRRELEVANTES (população < {POPULACAO_MINIMA} hab): {len(setores_irrelevantes):,}")
    print(f"      População: {setores_irrelevantes['v0001'].sum():,}")
    
    setores_criticos = gdf_mapa_setores[gdf_mapa_setores['categoria_bivariada'] == 'Alta + Baixo']
    print(f"\n    Setores CRÍTICOS (alta densidade, baixo acesso): {len(setores_criticos):,}")
    print(f"      População afetada: {setores_criticos['v0001'].sum():,}")
    
    setores_deserto = gdf_mapa_setores[gdf_mapa_setores['categoria_bivariada'] == 'Baixa + Baixo']
    print(f"\n    Setores DESERTO (baixa densidade, baixo acesso): {len(setores_deserto):,}")
    print(f"      População afetada: {setores_deserto['v0001'].sum():,}")